In [0]:
from datetime import datetime
from pyspark.sql.functions import col,lit , current_timestamp ,trim , concat_ws , md5 , when

In [0]:
dbutils.widgets.text("batch_id" , "1" , "Batch ID 1 , 2 OR 3")

In [0]:
batch_id = dbutils.widgets.get("batch_id")

print("batch_id = ", batch_id)


In [0]:
if batch_id == "1":
    print("batch_id is 1")
    dbutils.notebook.exit("for batch one there is no stagging layer - exiting")

In [0]:
spark.sql("USE CATALOG charles_schwab_retailbrokerage_dev_team_lemma")
spark.sql("create schema if not exists bronze")
spark.sql("use schema staging")

In [0]:
## initilixe variables

team_name = "team_lemma"
bronze_db = f"charles_schwab_retailbrokerage_dev_{team_name}.bronze"
silver_db = f"charles_schwab_retailbrokerage_dev_{team_name}.silver"
staging_db = f"charles_schwab_retailbrokerage_dev_{team_name}.staging"
Batch_Folder = f"Batch{batch_id}"


In [0]:
## Extract the carried run_id from the bronze_table

try:
    run_info_now = spark.sql(f"""
                               select _run_id , _batch FROM {bronze_db}.dailymarket 
                               where _batch = '{batch_id}'
                               LIMIT 1 
                            """).first()
    
    carried_run_id = run_info_now[0] if run_info_now[0] else "Unknown"
    carried_batch = run_info_now[1] if run_info_now[1] else batch_id
except Exception:
    carried_run_id = "Unknown"
    carried_batch = batch_id


print(f"bronze  : {bronze_db}")
print(f"staging : {staging_db}")
print(f"run_id  : {carried_run_id}")

In [0]:

staging_df = spark.table(f"{bronze_db}.dailymarket")\
                .filter(col("_batch") == batch_id)\
                .select(
                    trim(col("DM_DATE")).alias("DM_DATE"),
                    trim(col("DM_S_SYMB")).alias("DM_S_SYMB"),
                    trim(col("DM_CLOSE")).alias("DM_CLOSE"),
                    trim(col("DM_HIGH")).alias("DM_HIGH"),
                    trim(col("DM_LOW")).alias("DM_LOW"),
                    trim(col("DM_VOL")).alias("DM_VOL"),
                    trim(col("DM_ACTION")).alias("DM_ACTION"),
                )\
                .withColumn(
                    "row_hash" , md5(concat_ws("|",col("DM_DATE"),col("DM_S_SYMB"),col("DM_CLOSE"),col("DM_HIGH"),col("DM_LOW"),col("DM_VOL")))
                )
                

                          

In [0]:
silver_ready = spark.catalog.tableExists(
    f"charles_schwab_retailbrokerage_dev_{team_name}.silver.markethistory"
)

print(f"silver.markethistory exists: {silver_ready}")

In [0]:
if silver_ready:
    silver_mh =spark.table(f"{silver_db}.markethistory")\
                    .select(
                        col("dm_date").alias("s_date"),
                        col("dm_s_symb").alias("s_symb"),
                        md5(concat_ws("|",col("dm_date"),col("dm_s_symb"),col("dm_close"),col("dm_high"),col("dm_low"),col("dm_vol"))).alias("silver_hash")
    )
                    
    staged_df = staging_df.join(silver_mh,
                                (staging_df["DM_DATE"] == silver_mh["s_date"]) &
                                (staging_df["DM_S_SYMB"] == silver_mh["s_symb"]),
                                how="left"
                                ).withColumn("cdc_action" , 
                                             when(col("DM_ACTION") == "D", lit("D"))\
                                            .when(col("silver_hash").isNull()  , lit("N"))\
                                            .when(col("row_hash") != col("silver_hash") , lit("C"))\
                                            .otherwise(lit("X"))
                                )\
                                .drop("s_date" , "s_symb" , "silver_hash")
else:
    print(" silver.markethistory not found — all rows marked N")
    staged_df = staging_df.withColumn("cdc_action" , lit("N"))






    
                                

In [0]:
staged_final = staged_df.withColumn("_batch" , lit(carried_batch))\
                        .withColumn("_run_id" , lit(carried_run_id))\
                        .withColumn("_load_ts" , current_timestamp())

In [0]:
staged_final.write.format("delta")\
                  .mode("overwrite")\
                  .option("overwriteSchema" , "true")\
                  .saveAsTable(f"{staging_db}.dailymarket_current")

print(f"staging.dailymarket_current written for {Batch_Folder}")

In [0]:
result = spark.table(f"{staging_db}.dailymarket_current")\
              .groupBy("cdc_action")\
              .count()\
              .orderBy("cdc_action")

display(result)

In [0]:
total = spark.table(f"{staging_db}.dailymarket_current").count()
print(f"Total rows in staging.dailymarket_current: {Batch_Folder} : {total}")
print("Only N and C rows will be MERGEd into silver.markethistory")
print("D rows will be DELETEd from silver.markethistory")
print("X rows will be SKIPped")


In [0]:
%run ../../02_common_utils/operations

In [0]:

source_count = (
    spark.table(f"{bronze_db}.dailymarket")
    .filter(col("_batch") == batch_id)
    .count()
)
target_count = spark.table(f"{staging_db}.dailymarket_current").count()
carried_run_id = str(
    spark.table(f"{staging_db}.dailymarket_current").select("`_run_id`").first()[0]
)

log_pipeline_recon(
    spark=spark,
    run_id=carried_run_id,
    batch_id=Batch_Folder,
    domain="MARKET",
    table_name="dailymarket_current",
    source_layer="bronze",
    target_layer="staging",
    source_count=source_count,
    target_count=target_count
)

log_audit_event(
    spark=spark,
    run_id=carried_run_id,
    batch=Batch_Folder,
    layer="staging",
    table_name="dailymarket_current",
    operation="OVERWRITE",
    rows_affected=target_count
)


print(f"source_count : {source_count:,}")   
print(f"target_count : {target_count:,}")   